# Relational features for multiple-group BLE clustering

This notebook adds features that describe a source's *relationship to other simultaneously observed sources*. They are intended to separate Group 1 from Group 2 without using group labels, device names, payload identifiers, or absolute RSSI.

Run the feature cell after `CODE FIELD 2b` in a modelling notebook, on `int_train` and `int_eval` separately, and before balancing/binning. The function returns a packet-level dataframe with `rel_*` columns and a source-level table suitable for clustering.

> **Important:** this treats `source` as a stable identity during a capture. If BLE RPAs rotate quickly, use a source-session identifier (for example, source plus a time-session number) instead.

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter


In [ ]:
def add_relational_group_features(
    df: pd.DataFrame,
    source_col: str = 'source',
    time_col: str = 'time',
    window_seconds: float = 0.5,
    max_sources_per_window: int = 25,
    min_pair_windows: int = 2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add label-free source-neighbourhood features.

    Pairs from unusually busy windows are skipped: they are likely ambient traffic
    and otherwise create O(n^2) edges.  Use the returned source table for
    source-level clustering, or the returned packet table for the classifier.
    """
    required = {source_col, time_col}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    if window_seconds <= 0:
        raise ValueError('window_seconds must be positive')

    out = df.copy()
    out['_rel_window'] = np.floor(pd.to_numeric(out[time_col]) / window_seconds).astype('int64')
    source_windows = out[['_rel_window', source_col]].drop_duplicates()
    windows_per_source = source_windows.groupby(source_col).size().rename('rel_observed_windows')

    # One edge per unordered source pair and time window.
    pair_counts = Counter()
    eligible_windows = 0
    skipped_windows = 0
    for _, group in source_windows.groupby('_rel_window', sort=False):
        sources = np.sort(group[source_col].astype(str).unique())
        if len(sources) < 2:
            continue
        if len(sources) > max_sources_per_window:
            skipped_windows += 1
            continue
        eligible_windows += 1
        for left in range(len(sources) - 1):
            for right in range(left + 1, len(sources)):
                pair_counts[(sources[left], sources[right])] += 1

    source_index = windows_per_source.index.astype(str)
    stats = pd.DataFrame(index=source_index)
    stats.index.name = source_col
    stats['rel_observed_windows'] = windows_per_source.reindex(source_index).fillna(0).to_numpy()
    stats['rel_unique_neighbours'] = 0
    stats['rel_edge_weight_sum'] = 0.0
    stats['rel_affinity_mean'] = 0.0
    stats['rel_affinity_max'] = 0.0
    stats['rel_recurrent_neighbours'] = 0
    stats['rel_top_neighbour_share'] = 0.0
    stats['rel_neighbour_weight_entropy'] = 0.0

    neighbour_weights: dict[str, list[float]] = {source: [] for source in stats.index}
    neighbour_affinities: dict[str, list[float]] = {source: [] for source in stats.index}
    recurrent: Counter = Counter()
    for (left, right), shared_windows in pair_counts.items():
        # Jaccard affinity: co-observed windows / windows where either was observed.
        union_windows = stats.at[left, 'rel_observed_windows'] + stats.at[right, 'rel_observed_windows'] - shared_windows
        affinity = shared_windows / union_windows if union_windows else 0.0
        for source in (left, right):
            neighbour_weights[source].append(float(shared_windows))
            neighbour_affinities[source].append(affinity)
            if shared_windows >= min_pair_windows:
                recurrent[source] += 1

    for source in stats.index:
        weights = np.asarray(neighbour_weights[source], dtype=float)
        affinities = np.asarray(neighbour_affinities[source], dtype=float)
        if len(weights) == 0:
            continue
        total_weight = weights.sum()
        probs = weights / total_weight
        stats.at[source, 'rel_unique_neighbours'] = len(weights)
        stats.at[source, 'rel_edge_weight_sum'] = total_weight
        stats.at[source, 'rel_affinity_mean'] = affinities.mean()
        stats.at[source, 'rel_affinity_max'] = affinities.max()
        stats.at[source, 'rel_recurrent_neighbours'] = recurrent[source]
        stats.at[source, 'rel_top_neighbour_share'] = weights.max() / total_weight
        stats.at[source, 'rel_neighbour_weight_entropy'] = -(probs * np.log(probs)).sum()

    stats['rel_coactive_window_fraction'] = (
        stats['rel_edge_weight_sum'] / stats['rel_observed_windows'].clip(lower=1)
    )
    stats = stats.reset_index()
    out = out.merge(stats, left_on=source_col, right_on=source_col, how='left', validate='many_to_one')
    out = out.drop(columns='_rel_window')

    print(
        f'Relational features: {len(pair_counts):,} source-pairs from '
        f'{eligible_windows:,} eligible windows; skipped {skipped_windows:,} busy windows.'
    )
    return out, stats


In [ ]:
# Run independently for each capture: never compute a temporal graph across train/evaluation files.
# Insert this after CODE FIELD 2b in a modelling notebook.
int_train, relational_train_by_source = add_relational_group_features(
    int_train, window_seconds=0.5, max_sources_per_window=25, min_pair_windows=2
)
int_eval, relational_eval_by_source = add_relational_group_features(
    int_eval, window_seconds=0.5, max_sources_per_window=25, min_pair_windows=2
)

relational_columns = [c for c in int_train.columns if c.startswith('rel_')]
print(relational_columns)
relational_train_by_source.describe().T


## How to evaluate these features

For supervised modelling, leave the `rel_*` columns out of `dropped_columns`. Compare: (1) baseline features, (2) `rel_*` features only, and (3) both. Report macro-F1 and class-wise recall.

For source-level clustering, merge `relational_*_by_source` with ground truth only after clustering, exclusively for evaluation. Do not include `group_id`, `group_label*`, `labelled_device`, payload prefixes, or source addresses in `X`.

A grouped split remains useful, but there is a design choice: computing graph features from all unlabelled test-capture packets is a transductive setting. State this explicitly. For a strict inductive test, compute source features separately per temporal fold and only expose information available at that point in time.

In [ ]:
# Optional: only use this with genuinely synchronised multi-receiver data.
# Absolute RSSI is intentionally not used. The feature is the receiver-to-receiver pattern.
def add_multi_receiver_rssi_features(df, receiver_col, source_col='source', rssi_col='rssi'):
    if not {receiver_col, source_col, rssi_col}.issubset(df.columns):
        raise ValueError('receiver, source, or RSSI column is missing')
    temp = df[[source_col, receiver_col, rssi_col]].copy()
    temp['_rssi'] = pd.to_numeric(
        temp[rssi_col].astype(str).str.replace('dBm', '', regex=False).str.strip(), errors='coerce'
    )
    pivot = temp.pivot_table(index=source_col, columns=receiver_col, values='_rssi', aggfunc='median')
    features = pd.DataFrame(index=pivot.index)
    features['mr_rssi_receiver_range'] = pivot.max(axis=1) - pivot.min(axis=1)
    features['mr_rssi_receiver_std'] = pivot.std(axis=1).fillna(0)
    return df.merge(features.reset_index(), on=source_col, how='left', validate='many_to_one')

# Example (do not run for the current single-receiver captures):
# int_train = add_multi_receiver_rssi_features(int_train, receiver_col='receiver_id')
